# IT3051 – Telco Customer Churn Prediction
## Random Forest Model Implementation

This notebook implements the Random Forest contribution for the Telco Customer Churn classification task.
It uses the existing train/test split already prepared by the project and does not perform any new preprocessing.

### Scope
- Load the provided split data exactly as prepared by the project
- Inspect the data shapes and class distribution
- Train a baseline Random Forest model
- Tune the model with GridSearchCV
- Compare baseline and tuned results
- Analyze feature importance
- Save the final tuned model
- Verify the saved model loads and predicts correctly

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

%matplotlib inline
sns.set_style('whitegrid')
RANDOM_STATE = 42

## 1. Load Prepared Data

The project already contains the prepared train/test split. We use these files exactly as they are provided.

In [ ]:
X_train = pd.read_csv('../../../SplitData/X_train.csv')
X_test = pd.read_csv('../../../SplitData/X_test.csv')
y_train = pd.read_csv('../../../SplitData/y_train.csv').iloc[:, 0]
y_test = pd.read_csv('../../../SplitData/y_test.csv').iloc[:, 0]

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape :', y_test.shape)
print('\nTarget distribution in y_train:')
print(y_train.value_counts().sort_index().to_string())
print('\nTarget distribution in y_test:')
print(y_test.value_counts().sort_index().to_string())

## 2. Inspect Data

This step confirms that the same processed and split data is being used by the project. The target is already encoded as 0/1, so no additional preprocessing is performed.

Random Forest is a supervised ensemble learning method that builds many decision trees and combines their predictions to reduce variance and improve stability.

Random Forest is suitable for this churn prediction task because the dataset contains many tabular features with possible nonlinear relationships and interactions. It handles mixed numeric and encoded categorical features well and provides interpretable feature importance.

## 3. Baseline Random Forest

A baseline Random Forest is trained using the default model behavior with a fixed random state for reproducibility. This gives an initial performance reference before tuning.

In [ ]:
baseline_rf = RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200)
baseline_rf.fit(X_train, y_train)

y_pred_base = baseline_rf.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred_base)
baseline_precision = precision_score(y_test, y_pred_base, zero_division=0)
baseline_recall = recall_score(y_test, y_pred_base)
baseline_f1 = f1_score(y_test, y_pred_base)
baseline_cm = confusion_matrix(y_test, y_pred_base)

print('BASELINE RANDOM FOREST RESULTS')
print('Accuracy :', round(baseline_accuracy, 4))
print('Precision:', round(baseline_precision, 4))
print('Recall   :', round(baseline_recall, 4))
print('F1-score :', round(baseline_f1, 4))
print('\nConfusion Matrix:')
print(baseline_cm)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_base, target_names=['No Churn', 'Churn']))

## 4. Baseline Evaluation

The baseline model gives a reference point. It is useful because it shows what the model can do before tuning. A baseline model is not necessarily the best model, but it helps us evaluate whether tuning is actually beneficial.

## 5. Hyperparameter Tuning

Hyperparameters are the settings that control how a model learns. For Random Forest, examples include the number of trees (`n_estimators`), tree depth (`max_depth`), minimum samples needed to split a node (`min_samples_split`), minimum samples in a leaf (`min_samples_leaf`), and the number of features considered at each split (`max_features`).

Hyperparameter tuning is performed to search for a better configuration than the default settings. This helps improve generalization and can reduce overfitting.

GridSearchCV tries multiple parameter combinations using cross-validation and selects the settings with the best average validation performance. In this problem, the target is imbalanced, so the selected scoring metric is `f1` for the positive (churn) class. F1 is appropriate because it balances precision and recall, making it more informative than accuracy when the minority class is important.

In [ ]:
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_tuner = RandomForestClassifier(random_state=RANDOM_STATE)
grid_search = GridSearchCV(
    estimator=rf_tuner,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    refit=True,
)

grid_search.fit(X_train, y_train)
print('Best hyperparameters:', grid_search.best_params_)
print('Best cross-validation score:', round(grid_search.best_score_, 4))

## 6. Tuned Random Forest

The best model identified by GridSearchCV is trained and then evaluated on the held-out test set.

In [ ]:
tuned_rf = grid_search.best_estimator_
y_pred_tuned = tuned_rf.predict(X_test)

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_precision = precision_score(y_test, y_pred_tuned, zero_division=0)
tuned_recall = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)
tuned_cm = confusion_matrix(y_test, y_pred_tuned)

print('TUNED RANDOM FOREST RESULTS')
print('Accuracy :', round(tuned_accuracy, 4))
print('Precision:', round(tuned_precision, 4))
print('Recall   :', round(tuned_recall, 4))
print('F1-score :', round(tuned_f1, 4))
print('\nConfusion Matrix:')
print(tuned_cm)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_tuned, target_names=['No Churn', 'Churn']))

## 7. Baseline vs Tuned Comparison

A simple comparison helps us decide whether tuning improved the model on this particular dataset. The result should be interpreted honestly; tuning only counts as an improvement if the metrics actually improve.

In [ ]:
comparison_df = pd.DataFrame(
    {
        'Accuracy': [baseline_accuracy, tuned_accuracy],
        'Precision': [baseline_precision, tuned_precision],
        'Recall': [baseline_recall, tuned_recall],
        'F1-score': [baseline_f1, tuned_f1],
    },
    index=['Baseline Random Forest', 'Tuned Random Forest'],
)

print(comparison_df.round(4).to_string())

## 8. Feature Importance

Random Forest provides feature importance values. These values show how much each feature contributed to the model's predictive decisions. This can help us understand which variables matter most for churn prediction.

The table below is sorted from highest to lowest importance.

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': tuned_rf.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance.head(10).to_string(index=False))

plt.figure(figsize=(12, 8))
plt.barh(feature_importance.head(10)['Feature'][::-1], feature_importance.head(10)['Importance'][::-1])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 10 Random Forest Feature Importances')
plt.tight_layout()
plt.show()

## 9. Save Final Model

The tuned Random Forest model is saved as a new file in the project `models` folder. It is saved separately from the other model files and does not overwrite existing model artifacts.

In [ ]:
joblib.dump(tuned_rf, '../../../models/random_forest_final.pkl')
loaded_model = joblib.load('../../../models/random_forest_final.pkl')
print('Saved model loaded successfully.')
print('Loaded model predictions on first 3 test rows:', loaded_model.predict(X_test.head(3)).tolist())

## 10. Conclusions

The final Random Forest model was trained, tuned, evaluated, and saved using the existing project data. The notebook does not redo preprocessing or alter the train/test split, as required.

The evaluation metrics are reported exactly as produced by the model. If tuning did not improve the model, the comparison section reflects that honestly. Random Forest is useful for this problem because it captures nonlinear patterns and provides feature importance, but it is not always guaranteed to outperform other models on every metric.

### Limitations / observations
- The data is already processed and split; the model depends on that existing representation.
- The churn class is in the minority, so precision/recall trade-offs matter more than raw accuracy.
- Feature importance helps explain the model but does not prove causation.
- Tuning can improve or worsen the result depending on the data and evaluation metric.